# 词袋

上一章，我们解决了文字处理的第一个问题：如何把单个单词变成数字。现在我们面临第二个问题：**如何把一整句话（或一段话）变成数字？**

语言是序列数据，单词的意义离不开上下文。所以我们不能只看一个词，而要把整句话送进模型。

---

## 最直觉的想法：把索引编码拼成向量

最简单的做法是：把一句话里每个词的索引编码排成一列，当作输入向量。

比如「我去上学」的索引编码是 `[0, 8, 23, 48]`，直接把这四个数字作为输入。

这个方法的问题是：它依赖单词的**绝对位置**。"我去上学"和"我上学去"意思几乎相同，但索引编码向量分别是 `[0, 8, 23, 48]` 和 `[0, 23, 48, 8]`，两者只有 25% 相同。模型会把它们当成两件完全不同的事情。

## 词袋：忽略顺序，只看有没有

有一种更简单的思路：**不管词的顺序，只关心哪些词出现了**。这就是**词袋**（Bag of Words）。

顾名思义，把一句话里所有的词不分先后地扔进一个袋子：袋子里有哪些词，就是这句话的词袋表示。“我去上学”和“我上学去”的词袋完全相同，都是 {我, 去, 上, 学}。

词袋的编码方式是在单热编码的基础上稍作变化：不再只有一个位置是 1，而是**这句话里出现过的所有词对应的位置都是 1**。以词表大小为 50 为例：

```
"我去上学" 的词袋编码：

[1, 0, 0, 0, 0, 0, 0, 0, 1, 0,   <- 索引 0（我）和 8（去）是 1
 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,   <- 索引 23（上）是 1
 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]   <- 索引 48（学）是 1
```

无论词的顺序如何，「我去上学」和「我上学去」的词袋编码完全一样。

**词袋的优点与缺点**

**优点**：简单高效，消除了词序带来的干扰，对于只需判断**出现了哪些词**的任务（如情感分析、垃圾邮件过滤）效果很好。

**缺点**：完全丢弃了词序信息。“这部电影不好看”和“这部好看的电影不推荐”虽然情感截然相反，但如果词表重叠度高，词袋编码可能非常相似。对于需要理解语序的任务，词袋力不从心。

In [1]:
import csv
import re
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __init__(self):
        self.training = True

    def __call__(self, x: Tensor):
        return self.forward(x)

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### IMDB 数据集

在这一章，我们要训练一个能判断影评情感的模型：读一段观众写的影评，预测他是喜欢还是讨厌这部电影。

与上一章相比，`IMDBDataset` 有两处新变化：

**第一：影评用词袋编码**。把每条影评里出现过的所有词对应的位置置为 1，得到一个长度等于词表大小的向量。这就是这条影评的特征向量（`train_features` / `test_features`）。

**第二：划分训练集和测试集**。我们把最后 100 条影评留作测试集，其余作为训练集。模型只见过训练集，测试集用来检验它是否真正学到了规律，而不是把数据背下来了。

标签保持不变：1 表示喜欢，0 表示讨厌。

模型的输出是 (0, 1) 之间的一个小数（由 Sigmoid 保证），我们约定：
* 输出 **> 0.5**：预测为喜欢；
* 输出 **< 0.5**：预测为讨厌。

In [8]:
class IMDBDataset(Dataset):

    def __init__(self, filename):
        self.filename = filename
        super().__init__()

    def load(self):
        self.reviews = []
        self.sentiments = []
        with open(self.filename, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                self.reviews.append(self.clean_text(row[0].lower()).split())
                self.sentiments.append([0 if row[1] == "negative" else 1])

        self.vocabulary = sorted(set(word for line in self.reviews for word in line))
        self.word2index = {word: index for index, word in enumerate(self.vocabulary)}
        self.index2word = {index: word for index, word in enumerate(self.vocabulary)}
        self.tokens = [[self.word2index[word] for word in line if word in self.word2index] for line in self.reviews]

        # 词袋编码
        self.train_features = [self.bag_of_words(tokens) for tokens in self.tokens[:-100]]
        self.test_features = [self.bag_of_words(tokens) for tokens in self.tokens[-100:]]
        # 标签
        self.train_labels = self.sentiments[:-100]
        self.test_labels = self.sentiments[-100:]

    @staticmethod
    def clean_text(text):
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        return text

    def encode(self, text):
        words = self.clean_text(text.lower()).split()
        return [self.word2index[word] for word in words if word in self.word2index]

    def decode(self, tokens):
        return " ".join([self.index2word[index] for index in tokens])

    # 词袋编码：将一条影评的索引列表转换为单热向量（出现过的词置 1）
    def bag_of_words(self, tokens):
        vector = np.zeros(len(self.vocabulary))
        # 用 set() 去重，同一个词出现多次只算一次
        vector[list(set(tokens))] = 1
        return vector

    def onehot(self, index):
        vector = np.zeros(len(self.vocabulary))
        vector[index] = 1
        return vector

    @staticmethod
    def argmax(vector):
        return np.argmax(vector)

    # 评估函数
    def estimate(self, predictions):
        labels = np.array(self.labels)
        correct = (np.abs(predictions.data - labels) < 0.5).sum()
        return correct / len(labels)

## 模型

### 线性层

**kaiming 随机初始化**

从本章开始，我们在线性层使用kaiming 随机初始化权重。

Kaiming 初始化（又称 He Initialization）是何恺明等人在 2015 年提出的一种权重初始化策略，旨在解决深度神经网络训练中的**梯度消失**或**爆炸**问题。其核心在于根据每一层的输入神经元数量（$n_{in}$）来自适应地调整权重的方差：

$$
\sigma = \sqrt{\frac{2}{n_{in}}}
$$

通过维持前向传播信号和反向传播梯度的方差稳定性，Kaiming 初始化目前已成为配合 ReLU 类激活函数使用的行业标准。

**与 Xavier 初始化的区别**

Xavier 初始化（2010年）是针对 Tanh / Sigmoid 等饱和激活函数，公式为：

$$
\sigma = \sqrt{\frac{1}{n_{in}}}
$$

它假设激活函数在原点附近是线性的，这对 ReLU 并不成立。ReLU 的单侧截断特性让 Kaiming 初始化把系数从 1 提升到 2，精确补偿了 ReLU 的方差损耗，使深层网络的信号在每一层都保持健康的幅度，是现代深度学习能够稳定训练的重要基础之一。

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.data.shape}; bias{self.bias.data.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def train(self):
        super().train()
        for l in self.layers:
            l.train()

    def eval(self):
        super().eval()
        for l in self.layers:
            l.eval()

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### Sigmoid 激活函数层

Sigmoid 最适合用于**二分类问题的输出层**，此时我们正好需要一个介于 0 到 1 之间的概率值。

In [11]:
class Sigmoid(Layer):

    def __init__(self, clip_range=(-100, 100)):
        super().__init__()
        self.clip_range = clip_range

    def forward(self, x: Tensor):
        z = np.clip(x.data, self.clip_range[0], self.clip_range[1])
        a = Tensor(1 / (1 + np.exp(-z)))

        def gradient_fn():
            x.grad += a.grad * a.data * (1 - a.data)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 二元交叉熵损失函数

对应二元分类问题，我们使用**二元交叉熵**（Binary Cross-Entropy，BCE）作为损失函数。

In [12]:
class BCELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        clipped = np.clip(p.data, 1e-7, 1 - 1e-7)
        bce = Tensor(0 - np.sum(y.data * np.log(clipped) + (1 - y.data) * np.log(1 - clipped)) / len(y.data))

        def gradient_fn():
            p.grad += (clipped - y.data) / (clipped * (1 - clipped)) / len(y.data)

        bce.gradient_fn = gradient_fn
        bce.parents = {p}
        return bce

### 随机梯度下降优化器

In [13]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

In [14]:
class NNModel(Model):

    def train(self, dataset, epochs):
        self.layer.train()
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        self.layer.eval()
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [ ]:
LEARNING_RATE = 0.01

### 超参数：轮次

In [ ]:
EPOCHS = 100

### 建模

我们搭建一个两层的全连接网络：

- **隐藏层**：`Linear(150 → 32)`，输入是长度为 150 的词袋向量（词表大小），输出是 32 维的隐藏表示。这一层让模型学习哪些词的组合对情感判断有用；
- **输出层**：`Linear(32 → 1)`，把 32 维特征压缩成一个标量；
- **Sigmoid 激活函数**：把标量映射到 (0, 1) 区间，作为喜欢的概率。

为什么要加中间层，而不是直接从 150 映射到 1？因为直接映射只能学到线性关系（哪些词出现得多就打高分），而加了隐藏层之后，模型可以学到词之间的非线性组合关系，表达能力更强。

In [16]:
dataset = IMDBDataset('tinyimdb.csv')

layer = Sequential([Linear(len(dataset.vocabulary), 32),
                    Linear(32, 1),
                    Sigmoid()])
loss_fn = BCELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Linear[weight(32, 150); bias(32,)]
Linear[weight(1, 32); bias(1,)]
Sigmoid[]


### 迭代

In [ ]:
model.train(dataset, EPOCHS)

## 验证

### 测试

训练完毕，我们在**测试集**（模型从未见过的 100 条影评）上评估准确率：

In [17]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 100.00%


准确率达到了 100%。这在真实的大型数据集上几乎不可能，这里的 TinyIMDB 数据量很小、词汇也经过筛选，模型可以完全记住训练集的规律，并在同分布的测试集上表现完美。

即便如此，这个结果验证了一件重要的事：**词袋 + 全连接网络，已经可以有效处理情感分类这类不需要理解词序的任务**。垃圾邮件过滤、商品评论分类，词袋模型在这些场景中至今仍被广泛使用。

## 课后练习

词袋的概念也可以用于其他的分类数据集，比如用户购物记录、游客旅行记录等。思考一下，怎么把词袋用于这些场景？